# Занятие 1, демо 1. Три формы совпали. Этого достаточно?

Мы вывели три вычислительные формы одного оператора и убедились, что они дают
одно и то же. Вопрос: что именно это установило?

Дальше везде $S_0=0$ - начальное состояние нулевое. Это частный случай
оператора из условия: так отличие в маске ни с чем не смешивается.

In [ ]:
import torch

torch.set_num_threads(1)

"""Три формы causal linear attention и независимый эталон.

Тот же оператор, что в ДЗ-1. Здесь он нужен как материал демонстрации, а не
как задание, поэтому все три формы даны готовыми.
"""
import itertools

import torch


def reference_by_definition(q, k, v):
    """Определение оператора прямым скалярным суммированием, FP64.

    Матричных произведений и масок здесь нет вообще: это независимая точка
    отсчёта, с которой сравниваются все три формы.
    """
    q, k, v = (t.detach().to(torch.float64) for t in (q, k, v))
    B, H, T, d_k = q.shape
    d_v = v.shape[-1]
    y = torch.zeros(B, H, T, d_v, dtype=torch.float64)

    for b, h, t, p in itertools.product(range(B), range(H), range(T), range(d_v)):
        total = 0.0
        for i in range(t + 1):                      # i <= t: маска включающая
            dot = sum(float(q[b, h, t, a]) * float(k[b, h, i, a])
                      for a in range(d_k))
            total += dot * float(v[b, h, i, p])
        y[b, h, t, p] = total

    return y


def parallel(q, k, v, inclusive=True):
    T = q.shape[-2]
    mask = torch.ones(T, T, dtype=torch.bool, device=q.device)
    mask = mask.tril() if inclusive else mask.tril(diagonal=-1)
    scores = (q @ k.transpose(-1, -2)).masked_fill(~mask, 0.0)
    return scores @ v


def recurrent(q, k, v, inclusive=True):
    T = q.shape[-2]
    state = torch.zeros(*q.shape[:2], v.shape[-1], q.shape[-1],
                        dtype=q.dtype, device=q.device)
    outputs = []

    for t in range(T):
        if inclusive:                                # сначала запись, потом чтение
            state = state + v[..., t, :].unsqueeze(-1) * k[..., t, :].unsqueeze(-2)
            outputs.append((state @ q[..., t, :].unsqueeze(-1)).squeeze(-1))
        else:                                        # сначала чтение, потом запись
            outputs.append((state @ q[..., t, :].unsqueeze(-1)).squeeze(-1))
            state = state + v[..., t, :].unsqueeze(-1) * k[..., t, :].unsqueeze(-2)

    return torch.stack(outputs, dim=-2)


def chunkwise(q, k, v, chunk_size=2, inclusive=True):
    T = q.shape[-2]
    state = torch.zeros(*q.shape[:2], v.shape[-1], q.shape[-1],
                        dtype=q.dtype, device=q.device)
    outputs = []

    for a in range(0, T, chunk_size):
        b = min(a + chunk_size, T)
        qb, kb, vb = q[..., a:b, :], k[..., a:b, :], v[..., a:b, :]
        c = b - a
        mask = torch.ones(c, c, dtype=torch.bool, device=q.device)
        mask = mask.tril() if inclusive else mask.tril(diagonal=-1)
        inner = (qb @ kb.transpose(-1, -2)).masked_fill(~mask, 0.0)
        outputs.append(qb @ state.transpose(-1, -2) + inner @ vb)
        state = state + vb.transpose(-1, -2) @ kb

    return torch.cat(outputs, dim=-2)


def inputs(T=6, d_k=3, d_v=2, B=1, H=1, seed=0):
    g = torch.Generator().manual_seed(seed)
    mk = lambda *s: torch.randn(*s, generator=g, dtype=torch.float64)
    return mk(B, H, T, d_k), mk(B, H, T, d_k), mk(B, H, T, d_v)


def max_diff(a, b) -> float:
    return float((a.to(torch.float64) - b.to(torch.float64)).abs().max())

## Меняем ровно одну вещь

Маска включающая, $M_{ti}=\mathbf 1[i\leq t]$: в рекуррентной форме это порядок
«сначала запись $v_tk_t^\top$, потом чтение».

Строгая маска, $M_{ti}=\mathbf 1[i\lt t]$: «сначала чтение, потом запись».
Текущий токен перестаёт видеть сам себя.

Меняем во **всех трёх** формах одинаково и смотрим, согласны ли они между собой.

In [ ]:
q, k, v = inputs(T=7, d_k=3, d_v=2)          # 7 не кратно 3: последний блок неполный


def pairwise(inclusive):
    """Наибольшее расхождение трёх форм между собой."""
    args = dict(inclusive=inclusive)
    y_par = parallel(q, k, v, **args)
    y_rec = recurrent(q, k, v, **args)
    y_chunk = chunkwise(q, k, v, chunk_size=3, **args)
    return max(max_diff(y_par, y_rec), max_diff(y_par, y_chunk),
               max_diff(y_rec, y_chunk))


print(f"{'маска':<12} {'три формы между собой':>24}")
print(f"{'включающая':<12} {pairwise(True):>24.2e}")
print(f"{'строгая':<12} {pairwise(False):>24.2e}")
print()
print("На этом входе обе тройки согласованы в пределах погрешности FP64.")

## Теперь независимая точка отсчёта

Эталон вычисляет определение оператора напрямую: сумма по $i\leq t$, скалярно,
без матриц и без рекуррентности. Общего кода с тремя формами у него нет.

In [ ]:
y_ref = reference_by_definition(q, k, v)


def versus_reference(inclusive):
    args = dict(inclusive=inclusive)
    return max(max_diff(parallel(q, k, v, **args), y_ref),
               max_diff(recurrent(q, k, v, **args), y_ref),
               max_diff(chunkwise(q, k, v, chunk_size=3, **args), y_ref))


print(f"{'маска':<12} {'между собой':>14} {'против эталона':>17}")
print(f"{'включающая':<12} {pairwise(True):>14.2e} {versus_reference(True):>17.2e}")
print(f"{'строгая':<12} {pairwise(False):>14.2e} {versus_reference(False):>17.2e}")

## Почему так

Строгие версии согласованы **не случайно**. У всех трёх форм одна и та же
величина

$$
\tilde y_t=\sum_{i\lt t}(q_t^\top k_i)v_i,
$$

и отличается она от нужной ровно на собственный вклад токена:
$y_t-\tilde y_t=(q_t^\top k_t)v_t$. Три реализации, выведенные из одного
неверного определения, согласуются идеально - по построению, а не по удаче.

Короче всего это видно при $T=1$: правильный ответ $(q_1^\top k_1)v_1$,
строгий - ноль, потому что ни одного $i\lt 1$ не существует.

## Что установил запуск, а что нет

Согласие трёх форм показывает, что **на этих входах они согласованы между
собой**. Ни того, что они вычисляют нужную функцию, ни того, что они верны на
других входах, из этого не следует.

Эталон тоже ничего не доказывает. Он выведен из той же спецификации, только
другим путём, и снижает риск общей ошибки - но если спецификацию неверно прочли
оба раза, ошибутся все четыре реализации. Сильнее любого программного теста
здесь работает ручной пример на $T=1$: два числа, посчитанные на бумаге.

Отсюда практическое правило проверки: эталон пишется **независимо от
реализации и прямо из определения**, а на краевых случаях ответ считается
руками.